In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

/Users/souvikb/miniforge3/envs/spar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check MPS availability
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

✓ Using MPS (Apple Silicon GPU)


In [3]:
# 1. Loading Qwen model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # Use FP16 for speed
    device_map="auto"  # Auto-assigns to MPS
)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
# prompt = "What is 2+2?"
# inputs = tokenizer(prompt, return_tensors="pt").to("mps")
# outputs = model.generate(**inputs, max_new_tokens=20)
# print(tokenizer.decode(outputs[0]))


## LoRA on all layers

In [5]:
# 2. Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [6]:
# 3. Load dataset (small sample for testing)
dataset = load_dataset("gsm8k", "main", split="train[:50%]")
def format_example(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"


In [7]:
# 4. Training config optimized for M4 Max
training_args = SFTConfig(
    output_dir="./qwen-gsm8k-lora-all",
    num_train_epochs=1,
    per_device_train_batch_size=16,  # M4 Max can handle larger batches
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,  # Use FP16 for speed
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    report_to="none",
)

In [ ]:
# 5. Train (max_seq_length goes here)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    formatting_func=format_example,
    processing_class=tokenizer,
)

Truncating train dataset: 100%|██████████| 3736/3736 [00:00<00:00, 1220874.15 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [9]:
print("Starting training...")
trainer.train()

print("Saving model...")
trainer.save_model("./qwen-gsm8k-lora-all-final")

print("✓ Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


/Users/souvikb/miniforge3/envs/spar/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.587200
20,0.413000
30,0.329700
40,0.309700
50,0.304100
60,0.305300
70,0.329100
80,0.306200
90,0.307900
100,0.301900


Saving model...
✓ Training complete!


## LoRA on some selective layers

In [ ]:
# Using a fresh base model to make sure that we do not 
# run LoRA on top of already LoRA run model
base_id = "Qwen/Qwen2.5-1.5B-Instruct"

def load_base():
    return AutoModelForCausalLM.from_pretrained(base_id,dtype=torch.float16, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(base_id)
tokenizer.pad_token = tokenizer.eos_token
# 2. Configure late_LoRA

lora_config_late = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    layers_to_transform=list(range(14, 28)),
    layers_pattern="layers",
)
model_late = load_base()
model_late = get_peft_model(model_late, lora_config_late)
print("\n=== LATE LAYERS ONLY ===")
model_late.print_trainable_parameters() 


=== LATE LAYERS ONLY ===
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [ ]:
# 3. Load same dataset for late LoRA. Do not need to do it if it has already been downloaded
dataset = load_dataset("gsm8k", "main", split="train[:50%]")
def format_example(example):
    return {"text": f"Question: {example['question']}\nAnswer: {example['answer']}"}

dataset = dataset.map(format_example)

Map: 100%|██████████| 100/100 [00:00<00:00, 8722.32 examples/s]


In [12]:
# 4. Training config optimized for M4 Max
training_args = SFTConfig(
    output_dir="./qwen-gsm8k-lora-late",
    num_train_epochs=1,
    per_device_train_batch_size=16,  # M4 Max can handle larger batches
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,  # Use FP16 for speed
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    report_to="none",
)

In [ ]:

# 5. Train
trainer = SFTTrainer(
    model=model_late,
    args=training_args,
    train_dataset=dataset,
    formatting_func=format_example,
    processing_class=tokenizer,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [14]:
print("\nStarting training (late layers only)...")
trainer.train()

print("Saving model...")
trainer.save_model("./qwen-gsm8k-lora-late-final")

print("✓ Training complete!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



Starting training (late layers only)...


/Users/souvikb/miniforge3/envs/spar/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.659000
20,0.491200
30,0.386000
40,0.357700
50,0.345400
60,0.348400
70,0.368800
80,0.353600
90,0.352000
100,0.337200


Saving model...
✓ Training complete!


In [15]:
ckpt = "./qwen-gsm8k-lora-all-final"   # or late checkpoint
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForCausalLM.from_pretrained(ckpt, device_map="auto")
model.eval()

q = "What is 25 * 17?"
prompt = f"Question: {q}\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=8,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

gen_ids = out[0][inputs["input_ids"].shape[1]:]
gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

# keep only first line
final = gen_text.splitlines()[0]
print(final)   # should be: 425


25 * 17 =


In [25]:
## comparing the two checkpoints on a few examples

# GSM8K side-by-side eval for two checkpoints (deterministic, exact-match on final number)

import re
import gc
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- config ----
ALL_CKPT = "./qwen-gsm8k-lora-all-final"
LATE_CKPT = "./qwen-gsm8k-lora-late-final"
N_SAMPLES = 100  # Test on 100 problems
SEED = 42

# ---- helpers ----
def normalize_num_str(s: str):
    """Convert number string to normalized form (remove commas, handle int vs float)"""
    if s is None:
        return None
    s = s.replace(",", "").strip()
    try:
        v = float(s)
        if v.is_integer():
            return str(int(v))
        return str(v)
    except:
        return None

def extract_gold_num(answer_text: str):
    """Extract gold answer from GSM8K answer field (looks for #### marker)"""
    # GSM8K stores final answer after ####
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", answer_text)
    if m:
        return normalize_num_str(m.group(1))
    # fallback: last number in string
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", answer_text)
    return normalize_num_str(nums[-1]) if nums else None

def extract_pred_num(gen):
    # prefer GSM8K-style final marker if model emits it
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", gen)
    if m:
        return normalize_num_str(m.group(1))
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", gen)
    return normalize_num_str(nums[-1]) if nums else None

def build_prompt(q):
    # match training style
    return f"Question: {q}\nAnswer:"

@torch.no_grad()
def predict_one(model, tok, question):
    prompt = build_prompt(question)
    x = tok(prompt, return_tensors="pt").to(model.device)
    y = model.generate(
        **x,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = y[0][x["input_ids"].shape[1]:]   # robust
    gen = tok.decode(gen_ids, skip_special_tokens=True).strip()
    return gen, extract_pred_num(gen)


def evaluate_checkpoint(ckpt_path: str, questions, gold_nums):
    """Evaluate one checkpoint on all questions"""
    print(f"\nLoading checkpoint: {ckpt_path}")
    
    tok = AutoTokenizer.from_pretrained(ckpt_path)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(ckpt_path, device_map="auto")
    model.eval()

    correct = 0
    rows = []
    
    print(f"Evaluating {len(questions)} questions...")
    for i, (q, g) in enumerate(zip(questions, gold_nums)):
        gen, p = predict_one(model, tok, q)
        ok = (p == g) and (p is not None)
        correct += int(ok)
        
        # Save first 5 examples for inspection
        if i < 5:
            rows.append({
                "q": q[:60] + "...",  # Truncate long questions
                "gold": g,
                "pred": p,
                "raw_pred": gen[:80],  # First 80 chars of generation
                "correct": "✓" if ok else "✗"
            })
        
        # Progress indicator
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(questions)} done... (accuracy so far: {correct/(i+1):.2%})")

    acc = correct / len(questions)
    print(f"✓ Final accuracy: {acc:.2%} ({correct}/{len(questions)})")

    # Cleanup before loading next model
    del model
    del tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return acc, pd.DataFrame(rows)

# ---- Load test data ----
print("Loading GSM8K test set...")
gsm8k = load_dataset("gsm8k", "main", split="test")
gsm8k = gsm8k.shuffle(seed=SEED).select(range(N_SAMPLES))

questions = [x["question"] for x in gsm8k]
gold_nums = [extract_gold_num(x["answer"]) for x in gsm8k]

print(f"✓ Loaded {len(questions)} test questions")
print(f"\nExample question: {questions[0][:100]}...")
print(f"Example gold answer: {gold_nums[0]}")

# ---- Evaluate both checkpoints ----
print("\n" + "="*60)
print("EVALUATING ALL LAYERS")
print("="*60)
acc_all, sample_all = evaluate_checkpoint(ALL_CKPT, questions, gold_nums)

print("\n" + "="*60)
print("EVALUATING LATE LAYERS (14-27)")
print("="*60)
acc_late, sample_late = evaluate_checkpoint(LATE_CKPT, questions, gold_nums)

# ---- Summary ----
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)

summary = pd.DataFrame([
    {
        "Run": "All Layers (0-27)",
        "Checkpoint": ALL_CKPT,
        "GSM8K Accuracy": f"{acc_all:.1%}",
        "Correct": f"{int(acc_all * N_SAMPLES)}/{N_SAMPLES}",
    },
    {
        "Run": "Late Layers (14-27)",
        "Checkpoint": LATE_CKPT,
        "GSM8K Accuracy": f"{acc_late:.1%}",
        "Correct": f"{int(acc_late * N_SAMPLES)}/{N_SAMPLES}",
    },
])

print("\n")
display(summary)

print("\n" + "="*60)
print("SAMPLE PREDICTIONS - ALL LAYERS")
print("="*60)
display(sample_all)

print("\n" + "="*60)
print("SAMPLE PREDICTIONS - LATE LAYERS")
print("="*60)
display(sample_late)

# Save results
print("\n✓ Saving results to gsm8k_eval_results.csv")
summary.to_csv("gsm8k_eval_results.csv", index=False)


Loading GSM8K test set...
✓ Loaded 100 test questions

Example question: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's ...
Example gold answer: 109

EVALUATING ALL LAYERS

Loading checkpoint: ./qwen-gsm8k-lora-all-final
Evaluating 100 questions...
  20/100 done... (accuracy so far: 35.00%)
  40/100 done... (accuracy so far: 37.50%)
  60/100 done... (accuracy so far: 40.00%)
  80/100 done... (accuracy so far: 42.50%)
  100/100 done... (accuracy so far: 42.00%)
✓ Final accuracy: 42.00% (42/100)

EVALUATING LATE LAYERS (14-27)

Loading checkpoint: ./qwen-gsm8k-lora-late-final
Evaluating 100 questions...
  20/100 done... (accuracy so far: 40.00%)
  40/100 done... (accuracy so far: 37.50%)
  60/100 done... (accuracy so far: 38.33%)
  80/100 done... (accuracy so far: 41.25%)
  100/100 done... (accuracy so far: 40.00%)
✓ Final accuracy: 40.00% (40/100)

RESULTS SUMMARY




,Run,Checkpoint,GSM8K Accuracy,Correct
0,All Layers (0-27),./qwen-gsm8k-lora-all-final,42.0%,42/100
1,Late Layers (14-27),./qwen-gsm8k-lora-late-final,40.0%,40/100



SAMPLE PREDICTIONS - ALL LAYERS


,q,gold,pred,raw_pred,correct
0,Darrell and Allen's ages are in the ratio of 7...,109,10,Let x be the number of years ago when their ag...,✗
1,Lorraine and Colleen are trading stickers for ...,89,20,First find how many small stickers Lorraine tr...,✗
2,Indras has 6 letters in her name. Her sister's...,13,13,Half of the letters in Indras' name is 6/2 = <...,✓
3,Bethany can run 10 laps on the track in one ho...,5,5,Trey can run 10+4=<<10+4=14>>14 laps.\nShaelyn...,✓
4,An ice cream truck is traveling through a neig...,25,5,There were originally 5 children.\nOn the seco...,✗



SAMPLE PREDICTIONS - LATE LAYERS


,q,gold,pred,raw_pred,correct
0,Darrell and Allen's ages are in the ratio of 7...,109,7,Let x be the number of years ago when their ag...,✗
1,Lorraine and Colleen are trading stickers for ...,89,13,First find how many small stickers Lorraine tr...,✗
2,Indras has 6 letters in her name. Her sister's...,13,13,Half of the letters in Indras' name is 6/2 = <...,✓
3,Bethany can run 10 laps on the track in one ho...,5,5,"If Bethany can run 10 laps, then Trey can run ...",✓
4,An ice cream truck is traveling through a neig...,25,30,There were originally 5 children who chased th...,✗



✓ Saving results to gsm8k_eval_results.csv


In [ ]:
# MMLU eval cell: Full-LoRA vs Late-LoRA (few-shot optional, deterministic MCQ accuracy)

import re
import gc
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

ALL_CKPT = "./qwen-gsm8k-lora-all-final"
LATE_CKPT = "./qwen-gsm8k-lora-late-final"

SUBJECT = "all"          # keep "all" for overall score
SPLIT = "test[:500]"     # increase to "test" for full eval
N_SHOT = 0               # set to 5 if you want few-shot
SEED = 42

letters = ["A", "B", "C", "D"]

def build_example(ex, include_answer=True):
    txt = (
        f"Question: {ex['question']}\n"
        f"A. {ex['choices'][0]}\n"
        f"B. {ex['choices'][1]}\n"
        f"C. {ex['choices'][2]}\n"
        f"D. {ex['choices'][3]}\n"
        "Answer:"
    )
    if include_answer:
        txt += f" {letters[ex['answer']]}\n\n"
    return txt

def build_prompt(ex, fewshot_examples):
    header = "Select the correct option (A, B, C, or D).\n\n"
    shots = "".join(build_example(s, include_answer=True) for s in fewshot_examples)
    query = build_example(ex, include_answer=False)
    return header + shots + query

def parse_choice(gen_text):
    m = re.search(r"\b([ABCD])\b", gen_text.upper())
    return m.group(1) if m else None

@torch.no_grad()
def predict_choice(model, tok, prompt):
    x = tok(prompt, return_tensors="pt").to(model.device)
    y = model.generate(
        **x,
        max_new_tokens=6,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen_ids = y[0][x["input_ids"].shape[1]:]
    gen = tok.decode(gen_ids, skip_special_tokens=True).strip()
    return gen, parse_choice(gen)

def evaluate_checkpoint(ckpt_path, test_ds, dev_ds=None, n_shot=0):
    tok = AutoTokenizer.from_pretrained(ckpt_path)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(ckpt_path, device_map="auto")
    model.eval()

    correct = 0
    rows = []

    for i, ex in enumerate(test_ds):
        fewshots = []
        if n_shot > 0 and dev_ds is not None:
            # fixed few-shot slice for reproducibility
            fewshots = [dev_ds[j] for j in range(min(n_shot, len(dev_ds)))]

        prompt = build_prompt(ex, fewshots)
        raw, pred = predict_choice(model, tok, prompt)
        gold = letters[ex["answer"]]
        ok = (pred == gold)
        correct += int(ok)

        if i < 5:
            rows.append({
                "question": ex["question"][:70] + "...",
                "gold": gold,
                "pred": pred,
                "raw_pred": raw,
                "ok": ok
            })

    acc = correct / len(test_ds)

    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return acc, pd.DataFrame(rows)

# Load MMLU
test_ds = load_dataset("cais/mmlu", SUBJECT, split=SPLIT).shuffle(seed=SEED)
dev_ds = None
if N_SHOT > 0:
    dev_ds = load_dataset("cais/mmlu", SUBJECT, split="dev")

acc_all, sample_all = evaluate_checkpoint(ALL_CKPT, test_ds, dev_ds, n_shot=N_SHOT)
acc_late, sample_late = evaluate_checkpoint(LATE_CKPT, test_ds, dev_ds, n_shot=N_SHOT)

summary = pd.DataFrame([
    {"run": "full_lora", "checkpoint": ALL_CKPT, "mmlu_acc": acc_all, "n": len(test_ds), "n_shot": N_SHOT},
    {"run": "late_lora_14_27", "checkpoint": LATE_CKPT, "mmlu_acc": acc_late, "n": len(test_ds), "n_shot": N_SHOT},
])
summary["delta_vs_full"] = summary["mmlu_acc"] - acc_all

display(summary)
print("\nSamples (full lora):")
display(sample_all)
print("\nSamples (late lora):")
display(sample_late)
# Save results
print("\n✓ Saving results to mmlu_eval_results.csv")
summary.to_csv("mmlu_eval_results.csv", index=False)


Generating auxiliary_train split: 100%|██████████| 99842/99842 [00:00<00:00, 1441486.55 examples/s]


,run,checkpoint,mmlu_acc,n,n_shot,delta_vs_full
0,full_lora,./qwen-gsm8k-lora-all-final,0.446,500,0,0.000
1,late_lora_14_27,./qwen-gsm8k-lora-late-final,0.534,500,0,0.088



Samples (full lora):


,question,gold,pred,raw_pred,ok
0,The ________ perspective on sustainability re...,C,A,"A. Economic, Overuse",False
1,At which of the following locations does bile ...,B,B,B\n\nExplanation: The du,True
2,"The rise in business led, private regulation c...",C,C,"C. Proactive, Cost",True
3,Statement 1 | A homomorphism may have an empty...,B,NaN,Let's analyze each statement one,False
4,The set of integers Z with the binary operatio...,C,NaN,To determine the identity element of,False



Samples (late lora):


,question,gold,pred,raw_pred,ok
0,The ________ perspective on sustainability re...,C,A,A\n\nThe economic perspective on,False
1,At which of the following locations does bile ...,B,B,B\n\nThe duodenum,True
2,"The rise in business led, private regulation c...",C,D,D\n\nThe rise in business,False
3,Statement 1 | A homomorphism may have an empty...,B,D,D\n\nThe first statement is,False
4,The set of integers Z with the binary operatio...,C,NaN,To determine the identity element of,False


In [31]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Full
base_full = AutoModelForCausalLM.from_pretrained(BASE_ID, device_map="auto")
cfg_full = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model_full = get_peft_model(base_full, cfg_full)
model_full.print_trainable_parameters()

# Late
base_late = AutoModelForCausalLM.from_pretrained(BASE_ID, device_map="auto")
cfg_late = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    layers_to_transform=list(range(14, 28)),
    layers_pattern="layers",
)
model_late = get_peft_model(base_late, cfg_late)
model_late.print_trainable_parameters()


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
